In [1]:
import glob
import re
import os

import numpy as np
from pathlib import Path

from pymor.basic import *
from pymor.core.pickle import load

from RBInvParam.problems.elasticity.build import build_InstationaryModelIP

set_log_levels({
    'pymor' : 'WARN'
})

set_defaults({})


In [2]:
import matplotlib as mpl
import matplotlib.pyplot as plt

fontsize = 14
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "cm",
    "font.size": fontsize,
    #'text.latex.preamble': r'\usepackage{amsfonts} \usepackage{accents}',
    'figure.dpi': 200
})

In [3]:
from typing import Dict, Tuple, Optional

def get_last_file(path : Path) -> Path | None:
    files = []
    pattern = os.path.join(path, "TR_IRGNM_*.pkl")
    files += glob.glob(pattern)
    pattern = os.path.join(path, "FOM_IRGNM_*.pkl")
    files += glob.glob(pattern)

    indices = []
    for f in files:
        match = re.search(r'_*(?:TR|FOM)_IRGNM_(\d+)\.pkl$', f)
        if match:
            idx = int(match.group(1))
            indices.append((idx, f))
    
    if indices:
        _, max_file = max(indices, key=lambda x: x[0])
        
        return Path(max_file).name
    else:
        print("No matching files found.")

    return None

def filter_and_reorder(d: Dict, pattern: str = r'.*FOM.*') -> Tuple[Dict, Optional[str]]:
    regex = re.compile(pattern)
    matching = [k for k in d if regex.search(k)]
    non_matching = [k for k in d if k not in matching]

    # Sort keys alphabetically within each group
    non_matching_sorted = sorted(non_matching)
    matching_sorted = sorted(matching)

    # Build the reordered dict: alphabetically sorted non-matching first, then matching ones
    reordered = {k: d[k] for k in non_matching_sorted}
    reordered.update({k: d[k] for k in matching_sorted})

    # Return reordered dict and the single matching key (if exactly one match)
    if len(matching_sorted) == 1:
        return reordered, matching_sorted[0]
    else:
        return reordered, None

In [4]:
#SAVE_PATH = Path('/home/dealii/workdir/figs')
SAVE_PATH = Path('/home/benedikt/Schreibtisch/error_stagnation')

########################################################################################

WORK_DIR = Path('/home/benedikt/Dokumente/parabolische_inverse_probleme/experiments')
#WORK_DIR = Path('/home/dealii/workdir/experiments')
#WORK_DIR = Path('/home/benedikt/Dokumente/parabolische_inverse_probleme/experiments')

subset = 'longer_time_horizon'
obs_op = 'sensors'
data_dir_path = WORK_DIR / 'longer_time_horizon_'

experiment_names = []
pattern = re.compile(rf'^{subset}_.*_{obs_op}.*')

experiment_names += [
    d.name for d in data_dir_path.iterdir()
    if d.is_dir() and pattern.match(d.name)
]

data_paths = [data_dir_path / experiment_name for experiment_name in experiment_names]
file_names = [get_last_file(data_path) for data_path in data_paths]

########################################################################################

# data_dir_path = Path('/home/dealii/workdir/examples/elasticity/dumps') / '20251130_142544_TR_IRGNM'

# experiment_names = []
# pattern = re.compile(rf'.*')

# experiment_names += [
#     d.name for d in data_dir_path.iterdir()
#     if d.is_dir() and pattern.match(d.name)
# ]

# data_paths = [data_dir_path]
# file_names = [get_last_file(data_path) for data_path in data_paths]

########################################################################################

setup = None
data = {}
optimizer_parameters = {}

for (data_path, file_name) in zip(data_paths, file_names):            

    try:
        with open(data_path / file_name, 'rb') as file:
            data_ = load(file)
        data[str(data_path.name)] = data_
    except TypeError:
        print(f"Can not find dumps for {data_path}")
    except:
        print(f"Can not open {data_path / file_name}")

    if not setup:
        with open(data_path / 'setup.pkl', 'rb') as file:
            setup = load(file)

    
    optimizer_parameter_path = data_path / 'optimizer_parameter.pkl'
    with open(optimizer_parameter_path, 'rb') as file:
        optimizer_parameter = load(file)
        
    optimizer_parameters[str(data_path.name)] = optimizer_parameter

data, FOM_key = filter_and_reorder(data, pattern=r'.*FOM.*')
#assert FOM_key

# if not 'FOM' in locals():
#     FOM = build_InstationaryModelIP(setup=setup)

print(data.keys())


dict_keys(['longer_time_horizon_TR_sensors', 'longer_time_horizon_TR_sensors_time_step', 'longer_time_horizon_TR_sensors_time_step_HaPOD_1e-12', 'longer_time_horizon_FOM_sensors'])


In [5]:
import pandas as pd

rows = []
regex = re.compile('.*FOM.*')

columns = [
    'Algorithm', 
    'time [s]', 
    'speed up', 
    'FOM solves', 
    r'$n_Q$', 
    r'$n_V$',
    'o. iter',
    'total iter.'
]

FOM_key = 'new_baseline_FOM_sensors'
FOM_total_runtime = data[FOM_key]['total_runtime'][-1]
FOM_row = [
    FOM_key,
    FOM_total_runtime,
    '--',
    '--',
    '--',
    '--',
    len(data[FOM_key]['J'])-1,
    len(data[FOM_key]['J'])-1
]

for key, val in data.items():
    if key == FOM_key:
        continue
        
    # print(val['FOM_num_calls'])
    # FOM_solves = val['FOM_num_calls']['solve_state'] + \
    #              val['FOM_num_calls']['solve_adjoint'] + \
    #              val['FOM_num_calls']['solve_linearized_state'] + \
    #              val['FOM_num_calls']['solve_linearized_adjoint']
    FOM_solves = '--'

    speed_up = FOM_total_runtime / val['total_runtime'][-1]
    TR_Js = []
    inner_loop_statistics = val['inner_loop_statistics']
    for inner_loop_statistic in inner_loop_statistics:
        TR_Js += inner_loop_statistic['J']
    
    row = [
        key,
        int(val['total_runtime'][-1]),
        speed_up,
        FOM_solves,
        val['dim_Q_r'][-1],
        val['dim_V_r'][-1],
        len(val['J']),
        len(TR_Js)
    ]
        
    rows.append(row)

rows = [FOM_row] + rows
df = pd.DataFrame.from_records(rows, columns=columns)

KeyError: 'new_baseline_FOM_sensors'

In [7]:
df

,Algorithm,time [s],speed up,FOM solves,$n_Q$,$n_V$,o. iter,total iter.
0,new_baseline_FOM_sensors,10994.23035,--,--,--,--,25,25
1,new_baseline_TR_sensors,6857.00000,1.603199,--,10,575,10,20
2,new_baseline_TR_sensors_krylov,909.00000,12.089629,--,16,322,6,18
3,new_baseline_TR_sensors_time_step,552.00000,19.890956,--,79,257,5,19
4,new_baseline_TR_sensors_time_step_full,784.00000,14.015127,--,197,257,5,18


In [29]:
# import pandas as pd

# columns = ['Algorithm', 
#            'eps POD',
#            #'L2relerror',
#            #'H1relerror',
#            'time [s]', 
#            'speed up', 
#            'FOM solves', 
#            r'$n_Q$', 
#            r'$n_V$', 
#            'o. iter',
#            'total iter.']

# rows = []
# algo = 'FOM'
# FOM_solves = FOM_data['FOM_num_calls']['solve_state'] + FOM_data['FOM_num_calls']['solve_adjoint'] + \
#              FOM_data['FOM_num_calls']['solve_linearized_state'] + FOM_data['FOM_num_calls']['solve_linearized_adjoint']
# row = [algo,
#        '--',
#        #'--',
#        #'--',
#        int(FOM_data['total_runtime'][-1]),
#        '--',
#        FOM_solves,
#        '--',
#        '--',
#        len(FOM_data['J'])-1,
#        len(FOM_data['J'])-1]
       
# rows.append(row)

# #exp_name_ = '20250909_094613_TR_IRGNM'
# exp_names = ['20250909_094613_TR_IRGNM']
# #exp_names = [exp_name_ + id_ for id_ in ['1e-9', '1e-10', '1e-11', '1e-12', '1e-13', '1e-14']]


# for exp_name in exp_names:
#     with open(data_dir_path / exp_name / 'TR_IRGNM_final.pkl', 'rb') as file:
#         TR_data = load(file)

#     with open(data_dir_path / exp_name / 'optimizer_parameter.pkl', 'rb') as file:
#         optimizer_parameter = load(file)

#     #assert optimizer_parameter['enrichment']['parameter_HaPOD_tol'] == optimizer_parameter['enrichment']['state_HaPOD_tol']

#     algo = 'TR'
#     FOM_solves = TR_data['FOM_num_calls']['solve_state'] + TR_data['FOM_num_calls']['solve_adjoint'] + \
#                  TR_data['FOM_num_calls']['solve_linearized_state'] + TR_data['FOM_num_calls']['solve_linearized_adjoint']
    
#     HaPOD_tol = optimizer_parameter['enrichment']['state_HaPOD_tol']
#     #L2relerror_ = L2relerror(FOM_data['q'][-1], TR_data['q'][-1])
#     #H1relerror_ = H1relerror(FOM_data['q'][-1], TR_data['q'][-1])

#     TR_Js = []
#     inner_loop_statistics = TR_data['inner_loop_statistics']
#     for inner_loop_statistic in inner_loop_statistics:
#         TR_Js += inner_loop_statistic['J']
        
#     row = [algo,
#            f'{HaPOD_tol:.0e}',
#            #f'{L2relerror_:.2e}',
#            #f'{H1relerror_:.2e}',
#            int(TR_data['total_runtime'][-1]),
#            FOM_data['total_runtime'][-1] / TR_data['total_runtime'][-1],
#            int(FOM_solves),
#            TR_data['dim_Q_r'][-1],
#            TR_data['dim_V_r'][-1],
#            len(TR_data['J']),
#            len(TR_Js)]
            
#     rows.append(row)
    

# df = pd.DataFrame.from_records(rows, columns=columns)

In [ ]:
df

In [30]:
print(df.to_latex(index=False, float_format="%.2f", column_format='lc|cccccccc'))

\begin{tabular}{lc|cccccccc}
\toprule
Algorithm & eps POD & time [s] & speed up & FOM solves & $n_Q$ & $n_V$ & o. iter & total iter. \\
\midrule
FOM & -- & 33599 & -- & 3655 & -- & -- & 84 & 84 \\
TR & 1e-06 & 6235 & 5.39 & 301 & 11 & 918 & 11 & 176 \\
\bottomrule
\end{tabular}

